# A goal-oriented multi-agent workflow using CrewAI

## **Selected Use Case**: *SaaS Product Support – Technical Issue Diagnosis*

**Problem Statemen**t: SaaS companies receive technical queries related to login failures, feature usage, API errors, and integration issues.

**Suggested Agents**:

- Technical Issue Diagnosis Agent
- Product Knowledge Reasoning Agent (non-RAG)
- Troubleshooting Response Agent
- Engineering Escalation Agent

**Goal**: Reduce support workload while ensuring unresolved issues reach engineering teams.

In [ ]:
!pip install crewai crewai_tools langchain-openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of kubernetes to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB

**0. INSTALLATIONS & IMPORTS**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from crewai import Agent, Task, Crew, LLM

# --- API KEY CONFIGURATION ---
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    # Fallback for standard local execution or manual entry
    pass

CUSTOM_BASE_URL = "https://openai.vocareum.com/v1"

# Initialize the LLM using CrewAI's native wrapper or standard string format
llm = LLM(
    model="gpt-4o-mini",
    base_url=CUSTOM_BASE_URL,
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0
)

**1. MOCK KNOWLEDGE BASE (STATIC PRODUCT DOCS)**

In [ ]:
MOCK_SAAS_DOCS = """
=== SAAS KNOWLEDGE BASE & TROUBLESHOOTING RULES ===

1. CATEGORY: LOGIN & AUTHENTICATION
- Policy 1.1 (2FA Sync Error): If user receives 2FA/MFA errors after timezone changes, direct user to Settings > Time Sync or use 8-digit emergency backup codes.
- Policy 1.2 (Account Lockout): Accounts lock after 5 failed password attempts. Automatic reset occurs after 30 minutes, or user must click 'Forgot Password'.

2. CATEGORY: API & RATE LIMITS
- Policy 2.1 (429 Rate Limit Exceeded): Standard tier limits are 100 requests/minute. Users encountering 429 errors must implement exponential backoff retry headers in their client applications.
- Policy 2.2 (Invalid Token 401): API tokens expire every 90 days. Users must regenerate keys in Developer Portal > API Keys.

3. CATEGORY: WEBHOOKS & SYSTEM ERRORS
- Policy 3.1 (500 Internal Server Error / Database Deadlock): Max payload size supported by webhooks is 2MB. Payloads > 2MB causing 500 errors or DB deadlocks represent a core engine bug.
  RULE: Mark output as 'KNOWN BUG / NO LOCAL FIX' and escalate to Engineering immediately.
"""

**2. AGENT DEFINITIONS**

In [ ]:
diagnosis_agent = Agent(
    role="Technical Issue Diagnosis Specialist",
    goal="Analyze technical queries to determine error categories, severity, affected components, and root causes.",
    backstory="You are an expert in SaaS diagnostics, log analysis, and system architecture. You dissect user problem descriptions to identify precisely what failed.",
    llm=llm,
    verbose=True
)

knowledge_reasoning_agent = Agent(
    role="Product Knowledge Reasoning Specialist",
    goal="Apply static SaaS product policies, error-code lookups, and known workarounds to technical issues without using live RAG.",
    backstory="You maintain an exhaustive mental catalog of SaaS documentation, API specs, authentication rules, and known system limitations.",
    llm=llm,
    verbose=True
)

troubleshooting_response_agent = Agent(
    role="Troubleshooting Response Drafter",
    goal="Craft clear, professional, step-by-step troubleshooting guides or workaround instructions for customers.",
    backstory="You excel at translating complex technical fixes and error logs into empathetic, clear, and actionable instructions for users of all technical levels.",
    llm=llm,
    verbose=True
)

engineering_escalation_agent = Agent(
    role="Engineering Escalation Coordinator",
    goal="Evaluate whether a technical issue requires engineering intervention, bug tracking, or immediate tier-3 escalation.",
    backstory="You guard system integrity by rigorously routing unresolvable bugs, severe data corruption, unexpected 5xx API crashes, and security risks to core engineering.",
    llm=llm,
    verbose=True
)

**3. TASK DEFINITIONS**

In [ ]:
diagnose_issue_task = Task(
    description="""Analyze the technical support query: "{customer_query}".
Identify:
- Issue category (login_failure, feature_usage, api_error, integration_issue)
- Severity level (low, medium, high, critical)
- Affected system component (auth_service, dashboard_ui, webhook_engine, database)
- Technical summary of the symptoms.
Output as a structured summary.""",
    agent=diagnosis_agent,
    expected_output="A structured breakdown of the technical issue, category, severity, and affected component."
)

reason_knowledge_task = Task(
    description=f"""Based on the technical diagnosis above and the original query "{{customer_query}}",
cross-reference the issue against our static SaaS documentation below:

{MOCK_SAAS_DOCS}

Find and return the matching policy rule or solution reference.
If the problem exceeds local docs or represents a server crash/bug described above, return 'KNOWN BUG / NO LOCAL FIX'.""",
    agent=knowledge_reasoning_agent,
    expected_output="Matching static policy rule, documentation guidance, or 'KNOWN BUG / NO LOCAL FIX'."
)

draft_response_task = Task(
    description="""Using the diagnosis and retrieved product knowledge for: "{customer_query}",
create a friendly, structured troubleshooting response that:
- Acknowledges the specific technical issue
- Provides clear, numbered step-by-step remediation instructions
- Maintains a professional and reassuring tone""",
    agent=troubleshooting_response_agent,
    expected_output="A polished, user-facing troubleshooting response."
)

apply_escalation_logic = Task(
    description="""Evaluate whether the draft response is sufficient or if the issue must be escalated to engineering.
Escalate immediately if:
- Severity is 'critical' or 'high' (e.g., data loss, unhandled 5xx server errors, system crashes).
- The knowledge reasoning output is 'KNOWN BUG / NO LOCAL FIX'.
- The problem involves complex custom integration failures beyond standard documentation.

Output format:
- `resolved` if handled successfully with standard troubleshooting steps.
- `requires_engineering_escalation` with detailed escalation notes, error signatures, and priority level for the engineering queue.""",
    agent=engineering_escalation_agent,
    expected_output="Resolution status ('resolved' or 'requires_engineering_escalation') with justification."
)

**4. CREW SETUP & EXECUTION**

In [ ]:
saas_support_crew = Crew(
    agents=[
        diagnosis_agent,
        knowledge_reasoning_agent,
        troubleshooting_response_agent,
        engineering_escalation_agent
    ],
    tasks=[
        diagnose_issue_task,
        reason_knowledge_task,
        draft_response_task,
        apply_escalation_logic
    ],
    verbose=True
)

**5. Testing with prompt**

In [ ]:
customer_query = """We are receiving intermittent 500 Internal Server Errors and database deadlock exceptions whenever our webhook payload exceeds 5MB. This is breaking our production synchronization."""

print(f"\n--- PROCESSING QUERY ---\n{customer_query}\n")

# Use asynchronous kickoff since Jupyter already runs an active event loop
import asyncio

async def run_crew():
    result = await saas_support_crew.kickoff_async(inputs={"customer_query": customer_query})
    print("\n--- FINAL SYSTEM RESPONSE & ESCALATION STATUS ---\n")
    print(result)

# Execute inside the current event loop
await run_crew()


--- PROCESSING QUERY ---
We are receiving intermittent 500 Internal Server Errors and database deadlock exceptions whenever our webhook payload exceeds 5MB. This is breaking our production synchronization.



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 01c34cc6-29ff-4a87-a032-a872e9b402ca                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the technical support query: "We are receiving intermittent 500 Internal Server Errors and       │
│  database deadlock exceptions whenever our webhook payload exceeds 5MB. This is breaking our production         │
│  synchronization.".                                                                                             │
│  Identify:                                                                                                      │
│  - Issue category (login_failure, feature_usage, api_error, integration_issue)                                  │
│  - Severity level (low, medium, high, critical)                                                                 │
│  - Affected system component (auth_service, dashboard_ui, webhook_engine, database)                             │
│  - Technical summary of the symptoms.                                                                           │
│  Output as a structured summary.                                                                                │
│  ID: 1fd97c34-899d-4e52-bcce-0ce1574bd152                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Issue Diagnosis Specialist                                                                    │
│                                                                                                                 │
│  Task: Analyze the technical support query: "We are receiving intermittent 500 Internal Server Errors and       │
│  database deadlock exceptions whenever our webhook payload exceeds 5MB. This is breaking our production         │
│  synchronization.".                                                                                             │
│  Identify:                                                                                                      │
│  - Issue category (login_failure, feature_usage, api_error, integration_issue)                                  │
│  - Severity level (low, medium, high, critical)                                                                 │
│  - Affected system component (auth_service, dashboard_ui, webhook_engine, database)                             │
│  - Technical summary of the symptoms.                                                                           │
│  Output as a structured summary.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Engineering Escalation Coordinator                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  requires_engineering_escalation                                                                                │
│                                                                                                                 │
│  Justification: The issue is categorized as critical due to the intermittent 500 Internal Server Errors and     │
│  database deadlock exceptions occurring when the webhook payload exceeds 5MB. This is causing significant       │
│  disruptions in production synchronization. Additionally, the knowledge reasoning output indicates a 'KNOWN     │
│  BUG / NO LOCAL FIX', which necessitates immediate escalation to engineering for resolution. The combination    │
│  of server errors and database deadlocks suggests a deeper underlying issue that requires engineering           │
│  intervention.                                                                                                  │
│                                                                                                                 │
│  Error Signatures:                                                                                              │
│  - Intermittent 500 Internal Server Errors                                                                      │
│  - Database deadlock exceptions                                                                                 │
│                                                                                                                 │
│  Priority Level: Critical                                                                                       │
│                                                                                                                 │
│  Please escalate this issue to the engineering team for further investigation and resolution.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Evaluate whether the draft response is sufficient or if the issue must be escalated to engineering.      │
│  Escalate immediately if:                                                                                       │
│  - Severity is 'critical' or 'high' (e.g., data loss, unhandled 5xx server errors, system crashes).             │
│  - The knowledge reasoning output is 'KNOWN BUG / NO LOCAL FIX'.                                                │
│  - The problem involves complex custom integration failures beyond standard documentation.                      │
│                                                                                                                 │
│  Output format:                                                                                                 │
│  - `resolved` if handled successfully with standard troubleshooting steps.                                      │
│  - `requires_engineering_escalation` with detailed escalation notes, error signatures, and priority level for   │
│  the engineering queue.                                                                                         │
│  Agent: Engineering Escalation Coordinator                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 01c34cc6-29ff-4a87-a032-a872e9b402ca                                                                       │
│  Final Output: requires_engineering_escalation                                                                  │
│                                                                                                                 │
│  Justification: The issue is categorized as critical due to the intermittent 500 Internal Server Errors and     │
│  database deadlock exceptions occurring when the webhook payload exceeds 5MB. This is causing significant       │
│  disruptions in production synchronization. Additionally, the knowledge reasoning output indicates a 'KNOWN     │
│  BUG / NO LOCAL FIX', which necessitates immediate escalation to engineering for resolution. The combination    │
│  of server errors and database deadlocks suggests a deeper underlying issue that requires engineering           │
│  intervention.                                                                                                  │
│                                                                                                                 │
│  Error Signatures:                                                                                              │
│  - Intermittent 500 Internal Server Errors                                                                      │
│  - Database deadlock exceptions                                                                                 │
│                                                                                                                 │
│  Priority Level: Critical                                                                                       │
│                                                                                                                 │
│  Please escalate this issue to the engineering team for further investigation and resolution.                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- FINAL SYSTEM RESPONSE & ESCALATION STATUS ---

requires_engineering_escalation

Justification: The issue is categorized as critical due to the intermittent 500 Internal Server Errors and database deadlock exceptions occurring when the webhook payload exceeds 5MB. This is causing significant disruptions in production synchronization. Additionally, the knowledge reasoning output indicates a 'KNOWN BUG / NO LOCAL FIX', which necessitates immediate escalation to engineering for resolution. The combination of server errors and database deadlocks suggests a deeper underlying issue that requires engineering intervention.

Error Signatures:
- Intermittent 500 Internal Server Errors
- Database deadlock exceptions

Priority Level: Critical

Please escalate this issue to the engineering team for further investigation and resolution.
